# Traffic Sensor AI developer tutorial

This notebook is a hands-on tour of the current architecture. It shows how to set up the project, inspect the configuration, run the perception-period runtime, run the analytics-period runtime, and extend the repository through the public APIs.

The default cells use synthetic frames and synthetic detector/tracker adapters, so the tutorial can run without a camera, video file, model weights, CUDA, BoxMOT, or Ultralytics. Real-video cells are included later and are guarded by a flag.

## 1. Setup on a laptop or Jetson

Run these commands from the repository root in a terminal. Use one environment strategy, not both.

### Option A: Python virtual environment

Windows PowerShell:

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
python -m pip install -r requirements_deploy.txt
python -m pip install -e .
```

Linux or Jetson shell:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r requirements_deploy.txt
python -m pip install -e .
```

### Option B: Conda environment

```bash
conda create -n traffic_sensor python=3.10
conda activate traffic_sensor
python -m pip install --upgrade pip
python -m pip install -r requirements_deploy.txt
python -m pip install -e .
```

Jetson note: install PyTorch and Torchvision builds that match your JetPack/CUDA version before installing detector or tracker dependencies. The project keeps `torch` and `torchvision` commented in `requirements_deploy.txt` for that reason.

## 2. Architecture map

The repository is organized around four responsibilities:

```text
Vision I/O -> Perception -> PeriodObservationBatch -> Analytics
```

- `vision_io`: frame acquisition and production policies.
- `perception`: detector and tracker adapters.
- `analytics`: geometry and traffic-indicator domain logic.
- `runtime`: orchestration, timing, periods, continuity, buffers, and coordinate transformation.

The two processing stages remain useful as runtime concepts:

- Stage 1, implemented by `runtime.perception`, turns frames into a `PeriodObservationBatch`.
- Stage 2, implemented by `runtime.analytics`, turns a `PeriodObservationBatch` into per-area metric reports.

In [1]:
from pathlib import Path
import sys

# Make the notebook work whether it is opened from the repo root or from scripts/.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

for path in (ROOT, ROOT / "src", ROOT / "scripts"):
    value = str(path)
    if value not in sys.path:
        sys.path.insert(0, value)

print(ROOT)

C:\Users\hamza\Programs\traffic-sensor-ai


In [2]:
from config.load_build import build_areas, build_geometry_engine, load_config
from analytics.geometry.engine import GeometryEngine
from analytics.geometry.primitives import Area, Line, Polygon
from runtime.analytics import PeriodAnalyticsEngine
from runtime.continuity import AnalyticsContinuityContext, ContinuityPolicy
from runtime.observations import PeriodObservationBatch, PeriodObservationBuffer
from runtime.periods import resolve_period_policy
from runtime.perception import extract_tracking_outputs, process_perception_period
from runtime.timing import TimingPolicy
from perception.vision_io.frame_producer import Frame

print("Imports OK")

Imports OK


## 3. Configuration as the assembly contract

`config/traffic_metrics.yaml` is the runtime schema. It describes source timing, period sizing, continuity, homography, geometry, detector/tracker settings, and metric eligibility. The builder in `config.load_build` turns this YAML into typed runtime objects.

In [3]:
cfg_path = ROOT / "config" / "traffic_metrics.yaml"
cfg = load_config(str(cfg_path))

print("Top-level config sections:", sorted(cfg.keys()))
print("Configured detector:", cfg["detector"])
print("Coordinate policy:", cfg["geometry"].get("coordinate_space"))
print("Areas:", [area["area_id"] for area in cfg["geometry"].get("areas", [])])

Top-level config sections: ['analytics', 'continuity', 'detector', 'frame_processing', 'geometry', 'homography', 'input', 'metrics', 'timing', 'tracker']
Configured detector: {'model_name': 'yolo11n_finetuned', 'confidence': 0.5}
Coordinate policy: image
Areas: ['area_1', 'area_2', 'area_3', 'area_4', 'area_5']


In [4]:
# A compact tutorial config keeps this notebook independent of local videos and weights.
tutorial_cfg = {
    "input": {"source": "synthetic", "fps": 1.0},
    "timing": {"mode": "frame"},
    "analytics": {"period_seconds": 2.0, "max_obs_per_frame": 10},
    "continuity": {"enabled": True, "max_age_seconds": 30.0},
    "homography": {"enabled": False, "calibration_file": None},
    "detector": {"model_name": "synthetic", "confidence": 0.5},
    "tracker": {
        "method": "synthetic",
        "reid_model": None,
        "classes": None,
        "device": "cpu",
        "half": False,
        "per_class": False,
    },
    "geometry": {
        "coordinate_space": "image",
        "frame_size_reference": 100,
        "polygon_membership_mode": "center",
        "vicinity": {"image": 0.05, "world_m": 0.5},
        "lines": [{"line_id": "line_a", "points": [[0, 50], [100, 50]]}],
        "areas": [
            {
                "area_id": "area_1",
                "name": "tutorial lane",
                "area_type": "lane",
                "enabled": True,
                "flow_line_id": "line_a",
                "zone": {
                    "points": [[0, 0], [100, 0], [100, 100], [0, 100]],
                    "distance_meters": 100.0,
                },
            }
        ],
    },
    "metrics": {
        "flow": {
            "enabled": True,
            "spatial_scope": "any_area",
            "coordinate_requirement": "image_or_world",
            "counter": {"logic": "crossed"},
        },
        "density": {"enabled": False, "spatial_scope": "any_area", "coordinate_requirement": "world_required"},
        "space_occupancy": {"enabled": False, "spatial_scope": "any_area", "coordinate_requirement": "world_required"},
        "time_occupancy": {"enabled": True, "spatial_scope": "any_area", "coordinate_requirement": "image_or_world"},
        "space_headway": {"enabled": False, "spatial_scope": "lane_only", "coordinate_requirement": "world_required"},
        "time_headway": {"enabled": True, "spatial_scope": "lane_only", "coordinate_requirement": "image_or_world"},
        "speed": {"enabled": False, "spatial_scope": "any_area", "coordinate_requirement": "world_required"},
    },
}

period_policy = resolve_period_policy(tutorial_cfg, fps=tutorial_cfg["input"]["fps"])
areas = build_areas(tutorial_cfg, num_classes=2)
geometry_engine = build_geometry_engine(areas, tutorial_cfg)

print(period_policy)
print(areas[0])
print("line ids:", geometry_engine._line_ids)

PeriodPolicy(period_seconds=2.0, fps=1.0, max_obs_per_frame=10, max_observations=20)
area_id='area_1' name='tutorial lane' area_type='lane' enable=True description='' flow_line=Line(line_id='line_a', points=((0.0, 50.0), (100.0, 50.0))) zone=Polygon(polygon_id='area_1', points=[(0.0, 0.0), (100.0, 0.0), (100.0, 100.0), (0.0, 100.0)]) distance_meters=100.0 eligible_metrics=['flow', 'time_occupancy', 'time_headway'] ineligible_metrics={'density': 'disabled', 'space_occupancy': 'disabled', 'space_headway': 'disabled', 'speed': 'disabled'}
line ids: ['line_a']


## 4. Use the analytics geometry module directly

The geometry package is domain logic. It does not know about video sources, detectors, trackers, periods, or MQTT. It receives primitives plus arrays and returns spatial caches.

In [6]:
import numpy as np

points = np.asarray([[50.0, 49.0], [50.0, 51.0], [120.0, 51.0]], dtype=np.float32)
line_cache, polygon_cache = geometry_engine.compute(points)

print("distance shape:", line_cache["distance"].shape)
print("signs:", line_cache["sign"])
print("vicinity:", line_cache["vicinity_mask"])
print("inside area_1:", polygon_cache["area_1"])

distance shape: (1, 3)
signs: [[-1  1  1]]
vicinity: [[ True  True  True]]
inside area_1: [ True  True False]


## 5. Build a PeriodObservationBatch manually

`PeriodObservationBuffer` is the Stage 1 contract boundary. Perception writes canonical rows into it; analytics reads the frozen `PeriodObservationBatch`.

In [7]:
timing_policy = TimingPolicy.from_config(tutorial_cfg)
buffer = PeriodObservationBuffer(max_observations=period_policy.max_observations)

frames = [
    Frame(data=np.zeros((8, 8, 3), dtype=np.uint8), timestamp=1000.0, read_idx=0),
    Frame(data=np.zeros((8, 8, 3), dtype=np.uint8), timestamp=1001.0, read_idx=1),
]

for frame, point in zip(frames, points[:2]):
    timing = timing_policy.resolve(frame)
    buffer.append_frame(
        timing=timing,
        track_ids=np.asarray([10], dtype=np.int64),
        class_ids=np.asarray([0], dtype=np.int32),
        points=point.reshape(1, 2),
    )

manual_batch = buffer.freeze(period_idx=0, timing_mode=timing_policy.mode, fps=timing_policy.fps)

print("observations:", manual_batch.observation_count)
print("frames:", manual_batch.period_frame_ids)
print("timestamps:", manual_batch.timestamp)
print("points:\n", manual_batch.points)

observations: 2
frames: [0 1]
timestamps: [0. 1.]
points:
 [[50. 49.]
 [50. 51.]]


## 6. Run the perception-period runtime with synthetic adapters

`runtime.perception.process_perception_period` is the reusable Stage 1 loop. It orchestrates acquisition, timing, detector, continuous tracker, observation extraction, and buffering. The detector and tracker below are tiny test doubles that satisfy the same contracts as real adapters.

In [8]:
class SyntheticProducer:
    def __init__(self, frames):
        self.frames = list(frames)
        self.index = 0

    def start(self):
        return None

    def next_frame(self):
        if self.index >= len(self.frames):
            return None
        frame = self.frames[self.index]
        self.index += 1
        return frame

    def release(self):
        return None


class SyntheticDetector:
    class_names = ("car", "truck")
    num_classes = 2

    def detect_to_track(self, frame):
        return np.empty((0, 6), dtype=np.float32)


class SyntheticTracker:
    def __init__(self):
        self.calls = 0

    def update(self, detections, frame):
        bottom_y = 49.0 if self.calls == 0 else 51.0
        self.calls += 1
        return np.asarray([[48.0, bottom_y - 4.0, 52.0, bottom_y, 10.0, 0.95, 0.0]], dtype=np.float32)


producer = SyntheticProducer(frames)
detector = SyntheticDetector()
tracker = SyntheticTracker()
producer.start()

perception_result = process_perception_period(
    producer=producer,
    detector=detector,
    tracker=tracker,
    timing_policy=TimingPolicy.from_config(tutorial_cfg),
    period_seconds=period_policy.period_seconds,
    max_observations=period_policy.max_observations,
    period_idx=0,
    continuity_policy=ContinuityPolicy(enabled=True, max_age_seconds=30.0),
)

batch = perception_result.batch
print(perception_result)
print("batch points:\n", batch.points)

PerceptionPeriodResult(batch=PeriodObservationBatch(timestamp=array([0., 1.]), frame_id=array([0, 1], dtype=int64), track_id=array([10, 10], dtype=int64), class_id=array([0, 0]), points=array([[50., 49.],
       [50., 51.]], dtype=float32), is_context=array([False, False]), period_frame_ids=array([0, 1], dtype=int64), period_timestamps=array([0., 1.]), period_idx=0, timing_mode='frame', fps=1.0), frames_processed=2, end_of_stream=False, detection_seconds=2.2799998987466097e-05, tracking_seconds=3.789999755099416e-05)
batch points:
 [[50. 49.]
 [50. 51.]]


In [9]:
# Save the perception contract exactly as a developer would inspect it after Stage 1.
output_dir = ROOT / "outputs" / "tutorial"
output_dir.mkdir(parents=True, exist_ok=True)
batch_path = output_dir / f"period_{batch.period_idx:06d}_perception.npz"

np.savez_compressed(
    batch_path,
    timestamp=batch.timestamp,
    frame_id=batch.frame_id,
    track_id=batch.track_id,
    class_id=batch.class_id,
    points=batch.points,
    is_context=batch.is_context,
    period_frame_ids=batch.period_frame_ids,
    period_timestamps=batch.period_timestamps,
)

loaded = np.load(batch_path)
print(batch_path)
print(sorted(loaded.files))

C:\Users\hamza\Programs\traffic-sensor-ai\outputs\tutorial\period_000000_perception.npz
['class_id', 'frame_id', 'is_context', 'period_frame_ids', 'period_timestamps', 'points', 'timestamp', 'track_id']


## 7. Run the analytics-period runtime

`runtime.analytics.PeriodAnalyticsEngine` is the Stage 2 orchestrator. It resolves coordinate space, optionally transforms observations, asks `GeometryEngine` for spatial caches, derives events, and dispatches the vectorized metric estimators.

In [10]:
analytics_engine = PeriodAnalyticsEngine(
    areas=areas,
    geometry_engine=geometry_engine,
    cfg=tutorial_cfg,
    num_classes=detector.num_classes,
)
continuity = AnalyticsContinuityContext()

spatial = analytics_engine.compute_spatial(batch)
events = analytics_engine.derive_crossings(batch, spatial)
metrics = analytics_engine.compute_period(batch, continuity)

print("coordinate space:", spatial.coordinate_space)
print("crossed masks:\n", events.crossed_masks)
print("crossing payload:", events.crossings_by_area["area_1"])
print("period metrics:", metrics)

coordinate space: image
crossed masks:
 [[False  True]]
crossing payload: {'timestamp': array([0.5]), 'track_id': array([10], dtype=int64), 'class_id': array([0]), 'line_id': array(['line_a'], dtype=object)}
period metrics: {'area_1': {'flow': FlowResult(average_flow=0.5, average_flow_by_class=array([0.5, 0. ], dtype=float32)), 'time_occupancy': OccupancyResult(average_occupancy=1.0, current_occupancy=1.0), 'time_headway': TimeHeadwayResult(average_time_headway=None, current_time_headway=None)}}


## 8. Run the full pipeline on a real video

Use the CLI once your config points to valid model weights and tracker settings. This runs both runtime stages and prints each period report.

```bash
python scripts/run_sensor.py --source path/to/video.mp4 --config config/traffic_metrics.yaml --fps 30 --period-mins 1
```

To inspect only the perception-period dataset, use:

```bash
python scripts/run_perception.py --source path/to/video.mp4 --config config/traffic_metrics.yaml --output outputs/perception --max-periods 1
```

On Jetson, start with a short video and a small detector model, then tune `frame_processing`, `tracker.device`, `tracker.half`, and `analytics.max_obs_per_frame`.

In [ ]:
# Optional real-video example. Keep this False until your local paths and weights are ready.
RUN_REAL_VIDEO = False
VIDEO_PATH = ROOT / "inputs" / "sample.mp4"
CONFIG_PATH = ROOT / "config" / "traffic_metrics.yaml"

if RUN_REAL_VIDEO:
    from scripts.run_perception import run_perception

    exported = run_perception(
        config_path=str(CONFIG_PATH),
        output_dir=str(ROOT / "outputs" / "perception"),
        source=str(VIDEO_PATH),
        max_periods=1,
    )
    print(exported)
else:
    print("Set RUN_REAL_VIDEO = True after configuring VIDEO_PATH, model weights, and tracker dependencies.")

## 9. Module-by-module extension guide

### Add a detector backend

Implement `perception.detection.interface.IDetector` and return canonical detections with shape `(N, 6)`:

```text
[x1, y1, x2, y2, score, class_id]
```

Expose `class_names` and `num_classes` from the loaded model metadata. Then wire the adapter in `perception.detection.factory.build_detector`.

### Add a tracker backend

Keep the runtime-facing output compatible with `runtime.perception.extract_tracking_outputs`: at least seven columns with bbox, track id, score, and class id.

### Add a metric

Put the formula in `analytics.traffic_metrics.estimators`. Put result/data contracts in `analytics.traffic_metrics.models`. Add eligibility rules in `config.load_build.METRIC_CAPABILITIES`, then dispatch it in `runtime.analytics.PeriodAnalyticsEngine.compute_metrics`.

### Add geometry behavior

Keep primitives coordinate-system agnostic in `analytics.geometry.primitives`. Put image-space or world-space algorithms in `analytics.geometry.spatial`. Expose period-level caches through `analytics.geometry.engine.GeometryEngine.compute`.

### Add a frame producer

Add acquisition policy to `perception.vision_io.frame_producer`, then select it in `scripts.sensor_pipeline.build_frame_producer`. Keep producers focused on frame acquisition only.

## 10. Developer validation commands

Run these after changing modules:

```bash
python -m compileall config src scripts tests visualization
python -m unittest discover tests
```

Current configuration smoke check:

```bash
python -c "from config.load_build import load_config, build_areas, build_geometry_engine; from runtime.analytics import PeriodAnalyticsEngine; cfg=load_config('config/traffic_metrics.yaml'); areas=build_areas(cfg, num_classes=4); engine=build_geometry_engine(areas,cfg); analytics=PeriodAnalyticsEngine(areas,engine,cfg,num_classes=4); print(len(areas), len(engine._line_ids), analytics.coordinate_space, {a.area_id: sorted(a.eligible_metrics) for a in areas})"
```

Expected shape of the smoke output: number of areas, number of lines, selected coordinate mode, then eligible metrics by area.

## 11. Practical Jetson checklist

- Confirm the camera or video opens with OpenCV before running detector/tracker code.
- Install PyTorch/Torchvision builds for the exact JetPack/CUDA version.
- Use a small model first and increase model size only after latency is acceptable.
- Keep the tracker object alive across periods. The runtime already does this.
- Prefer `frame_processing.producer: realtime_simulation` for realistic prerecorded-video latency checks.
- Inspect exported perception batches before debugging metrics. Bad observations lead to bad period reports.
- Keep homography calibration under `homography`; metrics should never call homography directly.